In [1]:
import Pkg;
Pkg.activate(@__DIR__);
Pkg.instantiate();

  Activating new project at `~/ocrl`
  No Changes to `~/ocrl/Project.toml`
  No Changes to `~/ocrl/Manifest.toml`


In [7]:
using LinearAlgebra, PyPlot
import ForwardDiff as FD
using Test
import Convex as cvx 
import ECOS

using Random
Random.seed!(1)

using LaTeXStrings
using JupyterFormatter
enable_autoformat()

plt.rc("text", usetex = true)
plt.rc("font", family = "Times New Roman")
plt.rc("text.latex", preamble = "\\usepackage{{amsmath}}")

# Convex.jl tutorial

This is convex modeling tool in Julia that let's us write out problems in a simple way, and then Convex.jl transforms them and sends them off to be solved (we're using [ECOS](https://github.com/embotech/ecos) as our solver today). If you want examples/inspiration for this technology, there are a few like this:

- Python: [CVXPY](https://www.cvxpy.org/) or [CVXOPT](http://cvxopt.org/) (cvxpy is probably what you want)
- Matlab: [CVX](http://cvxr.com/cvx/) or [YALMIP](https://yalmip.github.io/) (I like CVX better)
- R: [CVXR](https://cvxr.rbind.io/)

For Convex.jl the [repo is here](https://github.com/jump-dev/Convex.jl), and the [docs are here](https://jump.dev/Convex.jl/stable/)

These tools are just used for formulating your problem and verifying that it is Convex. The problem itself is solved by one of many available solvers, many common ones are:

- OSQP
- ECOS 
- CPLEX 
- Mosek 
- Gurobi
- COSMO 
- SeDuMi 
- SDPT3 
- GLPK 
- Hypatia 

## Least Squares 
For overdetermined systems (more equations than variables, "skinny" matrix A)
$$ \begin{align} \min_{x} \quad & \|Ax - b\|^2_2
 \end{align}$$

In [12]:
@testset "overdetermined" begin
    # overdetermined
    A = randn(10, 5)
    b = randn(10)
    x = cvx.Variable(5)

    prob = cvx.minimize(cvx.sumsquares(A * x - b))  # sumsquares(y) = dot(y, y) = norm(y)^2
    cvx.solve!(prob, ECOS.Optimizer; silent_solver = false)

    xcvx = x.value::Matrix  # this will always be a matrix
    xcvx = vec(x.value)  # convert to vector easily

    # compare with pseudoinverse
    @test norm(xcvx - (A' * A \ (A' * b))) < 1e-4
end;

Test Summary:  | Pass  Total  Time
overdetermined |    1      1  0.0s

ECOS 2.0.8 - (C) embotech GmbH, Zurich Switzerland, 2012-15. Web: www.embotech.com/ECOS

It     pcost       dcost      gap   pres   dres    k/t    mu     step   sigma     IR    |   BT
 0  +0.000e+00  -1.322e+00  +7e+00  5e-01  6e-02  1e+00  2e+00    ---    ---    1  1  - |  -  - 
 1  -8.212e-02  -2.569e-01  +1e+00  1e-01  1e-02  2e-01  5e-01  0.7834  2e-02   1  1  1 |  0  0
 2  +8.555e-01  +1.222e+00  +1e+00  1e+00  5e-02  3e+00  3e-01  0.5729  6e-01   2  2  2 |  0  0
 3  +2.382e+00  +2.671e+00  +2e-01  1e-01  6e-03  6e-01  5e-02  0.8540  1e-02   2  1  1 |  0  0
 4  +7.104e-01  +2.170e+00  +1e-01  3e-01  1e-02  2e+00  4e-02  0.4921  5e-01   2  2  2 |  0  0
 5  +3.248e+00  +3.096e+00  +1e-01  7e-02  3e-03  3e-02  3e-02  0.5470  6e-01   2  2  2 |  0  0
 6  +4.724e+00  +4.713e+00  +1e-02  1e-02  5e-04  2e-02  3e-03  0.9009  4e-03   2  2  2 |  0  0
 7  +5.226e+00  +5.243e+00  +2e-03  3e-03  1e-04  3e-02  6e-04  0.8823  

For underdetermined systems (more variables than equations, "fat" matrix A)
$$ \begin{align} \min_{x} \quad & \|x\|^2_2 \\ 
 \text{st} \quad & A x = b 
 \end{align}$$

In [14]:
@testset "underdetermined" begin
    # overdetermined
    A = randn(5, 10)
    b = randn(5)
    x = cvx.Variable(10)

    prob = cvx.minimize(cvx.sumsquares(x))

    # add constraints
    prob.constraints += (A * x == b)
    cvx.solve!(prob, ECOS.Optimizer; silent_solver = false)

    xcvx = x.value::Matrix  # this will always be a matrix
    xcvx = vec(x.value)  # convert to vector easily

    # compare with pseudoinverse
    @test norm(xcvx - A' * ((A * A') \ b)) < 1e-4
end;

Test Summary:   | Pass  Total  Time
underdetermined |    1      1  0.2s

ECOS 2.0.8 - (C) embotech GmbH, Zurich Switzerland, 2012-15. Web: www.embotech.com/ECOS

It     pcost       dcost      gap   pres   dres    k/t    mu     step   sigma     IR    |   BT
 0  +0.000e+00  -1.322e+00  +6e+00  4e-01  7e-02  1e+00  2e+00    ---    ---    1  1  - |  -  - 
 1  -9.555e-02  -1.486e-01  +5e-01  3e-02  4e-03  1e-01  2e-01  0.9187  2e-02   1  1  1 |  0  0
 2  -1.315e-01  -1.136e-01  +5e-01  1e-01  7e-03  4e-01  2e-01  0.3838  5e-01   2  2  2 |  0  0
 3  +2.373e-01  +1.775e-01  +1e-01  2e-02  1e-03  3e-03  4e-02  0.9221  2e-01   2  1  2 |  0  0
 4  +2.708e-01  +2.648e-01  +1e-02  2e-03  1e-04  6e-04  3e-03  0.9142  1e-03   2  2  2 |  0  0
 5  +2.860e-01  +2.857e-01  +1e-03  3e-04  1e-05  5e-04  3e-04  0.9453  5e-02   2  2  2 |  0  0
 6  +2.873e-01  +2.872e-01  +2e-04  4e-05  2e-06  7e-05  4e-05  0.8726  9e-03   2  1  2 |  0  0
 7  +2.876e-01  +2.876e-01  +6e-06  2e-06  9e-08  5e-06  2e-06  0.9876

## Equality constrained QP 

$$ \begin{align} \min_{x} \quad & \frac{1}{2} x^TQx + q^Tx \\ 
 \text{st} \quad & A x = b 
 \end{align}$$

In [16]:
let
    n = 10
    Q = randn(n, n)
    Q = Q' * Q + I  # create PSD matrix
    q = randn(n)

    A = randn(3, n)
    b = randn(3)

    x = cvx.Variable(n)

    # NOTE: quadform(x, Q) = x'*Q*x
    cost = 0.5 * cvx.quadform(x, Q) + cvx.dot(q, x)  # dot() is fine

    prob = cvx.minimize(cost)
    prob.constraints += (A * x == b)

    cvx.solve!(prob, ECOS.Optimizer; silent_solver = false)

    xcvx = x.value::Matrix  # this will always be a matrix
    xcvx = vec(x.value)  # convert to vector easily
end;


ECOS 2.0.8 - (C) embotech GmbH, Zurich Switzerland, 2012-15. Web: www.embotech.com/ECOS

It     pcost       dcost      gap   pres   dres    k/t    mu     step   sigma     IR    |   BT
 0  +7.762e-01  -1.621e+01  +6e+01  4e-01  2e-01  1e+00  2e+01    ---    ---    1  2  - |  -  - 
 1  +6.058e-01  -2.939e-01  +6e+00  2e-02  1e-02  3e-01  2e+00  0.9178  3e-02   2  2  2 |  0  0
 2  +8.410e-01  +1.577e-02  +4e+00  4e-02  2e-02  1e+00  1e+00  0.5892  5e-01   2  2  2 |  0  0
 3  +1.227e+00  +1.118e+00  +5e-01  3e-03  2e-03  6e-02  1e-01  0.9356  6e-02   2  2  2 |  0  0
 4  +1.214e+00  +1.204e+00  +5e-02  2e-04  1e-04  4e-03  1e-02  0.9148  6e-04   2  2  2 |  0  0
 5  +1.209e+00  +1.208e+00  +7e-03  4e-05  2e-05  8e-04  2e-03  0.8871  4e-02   2  2  2 |  0  0
 6  +1.209e+00  +1.209e+00  +6e-04  4e-06  2e-06  2e-04  2e-04  0.9890  1e-01   3  2  2 |  0  0
 7  +1.209e+00  +1.209e+00  +4e-05  2e-07  1e-07  9e-06  1e-05  0.9421  3e-04   2  2  2 |  0  0
 8  +1.209e+00  +1.209e+00  +4e-06  2e-08  1e-

## Letting Convex.jl do the parsing 

$$ \begin{align} \min_{x} \quad & \|Ax - b\|_1 \\ 
 \text{st} \quad &\|x\|_2 \leq 3
 \end{align}$$
 
 This problem is not in any sort of "standard form", but it is convex. We will let Convex.jl will convert this into a standard form "canonicalizing it", and send it ECOS to solve. 

In [17]:
let
    A = randn(10, 5)
    b = randn(10)
    x = cvx.Variable(5)

    prob = cvx.minimize(norm(A * x - b, 1))
    prob.constraints += (norm(x, 2) <= 3)
    cvx.solve!(prob, ECOS.Optimizer; silent_solver = false)

    xcvx = x.value::Matrix  # this will always be a matrix
    xcvx = vec(x.value)  # convert to vector easily
end;


ECOS 2.0.8 - (C) embotech GmbH, Zurich Switzerland, 2012-15. Web: www.embotech.com/ECOS

It     pcost       dcost      gap   pres   dres    k/t    mu     step   sigma     IR    |   BT
 0  +5.411e-17  -3.000e+00  +6e+01  5e-01  6e-01  1e+00  3e+00    ---    ---    1  1  - |  -  - 
 1  +3.352e+00  +3.563e+00  +2e+01  7e-02  9e-02  9e-01  8e-01  0.8220  1e-01   1  1  1 |  0  0
 2  +3.993e+00  +4.039e+00  +4e+00  1e-02  2e-02  2e-01  2e-01  0.7878  4e-02   1  1  1 |  0  0
 3  +4.017e+00  +4.022e+00  +6e-01  2e-03  3e-03  3e-02  3e-02  0.8510  2e-02   1  1  1 |  0  0
 4  +4.037e+00  +4.037e+00  +2e-02  7e-05  1e-04  9e-04  1e-03  0.9667  2e-03   1  1  1 |  0  0
 5  +4.037e+00  +4.037e+00  +2e-04  8e-07  1e-06  1e-05  1e-05  0.9890  1e-04   1  1  1 |  0  0
 6  +4.037e+00  +4.037e+00  +3e-06  9e-09  1e-08  1e-07  1e-07  0.9890  1e-04   1  1  1 |  0  0
 7  +4.037e+00  +4.037e+00  +3e-08  1e-10  1e-10  1e-09  2e-09  0.9890  1e-04   1  0  0 |  0  0

OPTIMAL (within feastol=1.4e-10, reltol=7.5e-

## Convex Trajectory Optimization
$$ \begin{align} \min_{x_{1:N},u_{1:N-1}} \quad & \sum_{i=1}^{N-1} \bigg[ \|x_i - x_g\|_2^2 + \|u_i\|_1 \bigg] + \frac{1}{2}x_N^TQ_fx_N & \\ 
 \text{st} \quad & x_1 = x_{\text{IC}} \\ 
 & x_{i+1} = A x_i + Bu_i \quad &\text{for } i = 1,2,\ldots,N-1 \\ 
 & x_N = x_g \\ 
 & \|u_i\|_2 \leq 3 \quad &\text{for } i = 1,2,\ldots,N-1\\ 
 & x_{min} \leq x_i \leq x_{max} \quad &\text{for } i = 1,2,\ldots,N-1\\ 
 \end{align}$$

In [19]:
function controllable(A, B)
    n = size(A, 1)
    C = hcat([A^i * B for i = 0:(n-1)]...)
    return rank(C) == n
end

let

    # create linear system
    nx = 4
    nu = 2
    A = randn(nx, nx)
    B = randn(nx, nu)
    @assert controllable(A, B)

    # time steps 
    N = 20
    x_ic = randn(nx)
    x_g = randn(nx)

    # terminal cost 
    Qf = randn(nx, nx)
    Qf = Qf' * Qf + I # make PSD Qf 

    # create cvx variables x_k = X[:,k], u_k = U[:,k]
    X = cvx.Variable(nx, N)
    U = cvx.Variable(nu, N - 1)

    # create cost 
    cost = 0
    for k = 1:(N-1)
        xk = X[:, k]
        uk = U[:, k]
        cost += cvx.sumsquares(xk - x_g)
        cost += norm(uk, 1)
    end
    xn = X[:, N]
    cost += 0.5 * cvx.quadform(xn, Qf)

    # initialize cvx problem 
    prob = cvx.minimize(cost)

    # initial condition constraint 
    prob.constraints += X[:, 1] == x_ic

    for k = 1:(N-1)
        # dynamics constraints 
        prob.constraints += (X[:, k+1] == A * X[:, k] + B * U[:, k])
    end

    # goal constraint 
    prob.constraints += X[:, N] == x_g

    # norm(u)<3 
    for k = 1:(N-1)
        uk = U[:, k]
        prob.constraints += norm(uk, 2) <= 3
    end

    x_min = -20 * ones(nx)
    x_max = 20 * ones(nx)
    for k = 1:N
        xk = X[:, k]
        prob.constraints += xk <= x_max
        prob.constraints += xk >= x_min
    end

    # solve problem (silent solver tells us the output)
    cvx.solve!(prob, ECOS.Optimizer; silent_solver = false)

    if prob.status != cvx.MathOptInterface.OPTIMAL
        error("Convex.jl problem failed to solve for some reason")
    end

    # convert the solution matrices into vectors of vectors 
    X = X.value::Matrix
    U = U.value::Matrix
end;


ECOS 2.0.8 - (C) embotech GmbH, Zurich Switzerland, 2012-15. Web: www.embotech.com/ECOS

It     pcost       dcost      gap   pres   dres    k/t    mu     step   sigma     IR    |   BT
 0  +0.000e+00  -3.490e+03  +4e+03  6e-02  5e-01  1e+00  1e+01    ---    ---    1  2  - |  -  - 
 1  +3.196e+01  -1.350e+03  +2e+03  2e-02  2e-01  1e+00  5e+00  0.6371  9e-02   1  1  1 |  0  0
 2  +6.284e+01  -1.298e+03  +2e+03  2e-02  1e-01  2e+00  5e+00  0.2086  7e-01   2  2  2 |  0  0
 3  +8.851e+01  -1.191e+03  +2e+03  2e-02  1e-01  2e+00  5e+00  0.2525  7e-01   2  2  2 |  0  0
 4  +1.610e+02  -6.469e+02  +9e+02  1e-02  5e-02  3e+00  3e+00  0.9890  6e-01   2  1  1 |  0  0
 5  +1.146e+02  -2.958e+02  +5e+02  6e-03  2e-02  2e+00  1e+00  0.5473  1e-01   2  1  2 |  0  0
 6  +1.204e+02  -2.971e+02  +5e+02  6e-03  2e-02  2e+00  1e+00  0.0775  9e-01   2  2  2 |  0  0
 7  +9.883e+01  +5.683e+00  +1e+02  1e-03  3e-03  4e-01  3e-01  0.9890  2e-01   2  1  2 |  0  0
 8  +9.752e+01  +7.894e+01  +2e+01  3e-04  5e-